# Chapter 7 assignment: PINNs for integral equations

**Learning goals**

1. Build the quadrature-based PINN loss for a Fredholm equation of the second kind and check it against a manufactured solution.
2. See that, with a fixed quadrature rule, a PINN converges at best to the solution of the *discretized* equation (Remark "PINNs and the Nyström method" in the book).
3. Compare accuracy and cost with the Nyström method.
4. Replace a Volterra integral by auxiliary outputs (A-PINN) when the kernel depends on $x$ (Exercise "Volterra kernel depending on x").

Everything runs on a CPU in a few minutes. Cells marked `# TODO` are for you; the cells headed **Check your answer** test your results.

In [ ]:
import sys, time, pathlib
import numpy as np
import torch

# make the companion library importable when the notebook is run from repo/notebooks
root = pathlib.Path.cwd()
if (root / "pinnode").exists() is False and (root.parent / "pinnode").exists():
    root = root.parent
sys.path.insert(0, str(root))

from pinnode import MLP, ConstrainedModel, derivative, to_tensor, train, rel_l2, linf, gauss_legendre
from pinnode.reference import nystrom_fredholm, volterra_trapezoid
from pinnode.utils import configure_torch, set_seed

configure_torch("float64", threads=1)
set_seed(1234)
print("torch", torch.__version__)

## Part 1: a Fredholm equation with a smooth, non-degenerate kernel

We solve
$$u(x) = f(x) + \lambda \int_0^1 e^{x t}\, u(t)\, \mathrm{d}t, \qquad x\in[0,1],\quad \lambda = 0.25,$$
with $f$ chosen such that the exact solution is $u^\ast(x) = e^{-x}$. Since
$\int_0^1 e^{xt} e^{-t}\,\mathrm{d}t = (e^{x-1}-1)/(x-1)$ (with the value 1 at $x=1$),
$$f(x) = e^{-x} - \lambda\, \frac{e^{x-1}-1}{x-1}.$$

**Theory recap.** Here $\kappa = \max_x \int_0^1 e^{xt}\,\mathrm{d}t = e-1$, so $|\lambda|\kappa \approx 0.43 < 1$: the Neumann series converges and $\|u_\theta - u\|_\infty \le \|r\|_\infty / (1-|\lambda|\kappa)$ for any approximation $u_\theta$ with residual $r$.

In [ ]:
LAM = 0.25

def u_exact(x):
    return np.exp(-np.asarray(x, float))

def phi(x):
    # (e^{x-1} - 1)/(x - 1), continuous at x = 1
    x = np.asarray(x, float)
    out = np.ones_like(x)
    m = np.abs(x - 1.0) > 1e-8
    out[m] = np.expm1(x[m] - 1.0) / (x[m] - 1.0)
    return out

def f_rhs(x):
    return u_exact(x) - LAM * phi(x)

def kernel(x, t):
    return np.exp(x * t)

x_test = np.linspace(0.0, 1.0, 1001)

### Task 1.1: the residual with Gauss–Legendre quadrature

Complete `make_loss`: with nodes $t_j$ and weights $w_j$ of the $Q$-point Gauss–Legendre rule on $[0,1]$ (use `gauss_legendre(Q, 0.0, 1.0)`), the residual at the collocation points is
$$R_i = u_\theta(x_i) - f(x_i) - \lambda \sum_{j=1}^{Q} w_j K(x_i,t_j)\,u_\theta(t_j).$$
Precompute the matrix $\mathsf{K}_{ij} = w_j K(x_i, t_j)$ once, outside the loss.

In [ ]:
def make_loss(model, x_col, Q):
    tq, wq = gauss_legendre(Q, 0.0, 1.0)
    Kmat = torch.as_tensor(wq[None, :] * kernel(x_col[:, None], tq[None, :]))   # (M, Q)
    xc, tq_t = to_tensor(x_col), to_tensor(tq)
    f_c = to_tensor(f_rhs(x_col))

    def loss_fn():
        # TODO: evaluate the network at the collocation points and at the quadrature nodes,
        #       form the residual R (shape (M, 1)) and return (mean(R^2), {"residual": mean(R^2)})
        raise NotImplementedError

    return loss_fn

### Task 1.2: train and compare with the exact solution

Train an MLP (tanh, 2 hidden layers of 20 neurons, inputs scaled to $[0,1]$) on $M=64$ uniform collocation points with $Q=8$: 2000 Adam iterations (learning rate $10^{-2}\to10^{-4}$) followed by at most 300 L-BFGS iterations.

In [ ]:
def train_fredholm(Q, adam_iters=2000, lbfgs_iters=300, seed=1234):
    torch.manual_seed(seed)
    model = MLP(1, 1, width=20, depth=2, activation="tanh", domain=(0.0, 1.0))
    x_col = np.linspace(0.0, 1.0, 64)
    hist = train(model.parameters(), make_loss(model, x_col, Q), adam_iters=adam_iters, lr=1e-2,
                 lr_final=1e-4, lbfgs_iters=lbfgs_iters, print_every=0, verbose=False)
    with torch.no_grad():
        u = model(to_tensor(x_test)).numpy().ravel()
    return u, hist

u8, hist8 = train_fredholm(Q=8)
err8 = rel_l2(u8, u_exact(x_test))
print(f"Q=8: relative L2 error {err8:.2e}, training time {hist8.train_time:.1f} s")

**Check your answer.** The relative $L^2$ error should be below $10^{-3}$ (typically $10^{-5}$–$10^{-4}$).

In [ ]:
assert err8 < 1e-3, f"error {err8:.2e} too large: check the residual (signs, weights, lambda)"
print("Task 1.2 passed")

### Task 1.3: the Nyström method

Use `nystrom_fredholm(kernel, f_rhs, LAM, 0.0, 1.0, n)` (it returns a callable Nyström interpolant) with $n = 8$ and time it. Compute the ratio of the PINN training time to the Nyström time, and the two errors.

In [ ]:
# TODO: compute u_nys on x_test with n = 8, its relative L2 error err_nys and its run time t_nys
t0 = time.perf_counter()
u_nys = None   # TODO
t_nys = time.perf_counter() - t0
err_nys = None  # TODO
print(f"Nystrom (n=8): error {err_nys}, time {t_nys:.2e} s; PINN/Nystrom time ratio {hist8.train_time / t_nys:.1e}")

**Check your answer.** The Nyström error should be below $10^{-8}$; the time ratio is several orders of magnitude.

In [ ]:
assert err_nys is not None and err_nys < 1e-8
print("Task 1.3 passed")

### Task 1.4: the quadrature limits the PINN

For $Q = 1, 2, 3$, train the PINN and compute the Nyström solution **with the same rule** ($n=Q$). Tabulate the PINN error, the Nyström error, and the distance between the two solutions. Explain your observations with the Remark "PINNs and the Nyström method": what does the PINN converge to? (Reduce `adam_iters` if you are short of time.)

In [ ]:
rows = []
for Q in (1, 2, 3):
    # TODO: train the PINN with Q nodes, compute the Nystrom interpolant with n = Q,
    #       and append (Q, error PINN, error Nystrom, rel_l2(PINN, Nystrom)) to rows
    pass
for r in rows:
    print("Q=%d  PINN %.2e  Nystrom %.2e  |PINN-Nystrom| %.2e" % r)

**Check your answer.** For $Q=1$ both errors should be of the same size (around $10^{-2}$), while the distance between the PINN and the one-node Nyström solution is much smaller than either error.

In [ ]:
if rows:
    q1 = rows[0]
    assert q1[3] < 0.2 * q1[1], "for Q=1 the PINN should be close to the discrete (Nystrom) solution"
    print("Task 1.4 passed")

## Part 2: an A-PINN for a Volterra kernel that depends on $x$

$$u(x) = 1 - \int_0^x (x-t)\,u(t)\,\mathrm{d}t, \qquad x\in[0,b],\quad b = \pi, \qquad u^\ast(x) = \cos x.$$

With $A(x) = \int_0^x (x-t)u(t)\,\mathrm{d}t$, Leibniz' rule gives $A' = B$ with $B(x) = \int_0^x u(t)\,\mathrm{d}t$, and $B' = u$. Hence three residuals,
$$R_1 = u - 1 + A, \qquad R_2 = A' - B, \qquad R_3 = B' - u,$$
and the initial values $u(0)=1$, $A(0)=0$, $B(0)=0$, which we build into the outputs:
$u_\theta = 1 + x N_1$, $A_\theta = x N_2$, $B_\theta = x N_3$. (The condition $A'(0)=0$ need not be imposed: it follows from $R_2=0$ and $B(0)=0$.)

In [ ]:
B_END = np.pi
x_test2 = np.linspace(0.0, B_END, 1001)

def build_apinn(seed=1234):
    torch.manual_seed(seed)
    net = MLP(1, 3, width=32, depth=3, activation="tanh", domain=(0.0, B_END))
    # TODO: return ConstrainedModel(net, transform) whose three output columns are
    #       u = 1 + x N_1,  A = x N_2,  B = x N_3
    raise NotImplementedError

def apinn_loss(model, x_col):
    x = to_tensor(x_col, requires_grad=True)
    def loss_fn():
        out = model(x)
        u, A, B = out[:, :1], out[:, 1:2], out[:, 2:3]
        # TODO: residuals R1, R2, R3 (use derivative(A, x) and derivative(B, x));
        #       return the sum of their mean squares and a dict with the three parts
        raise NotImplementedError
    return loss_fn

In [ ]:
model2 = build_apinn()
hist2 = train(model2.parameters(), apinn_loss(model2, np.linspace(0.0, B_END, 201)), adam_iters=4000, lr=5e-3,
              lr_final=5e-5, lbfgs_iters=500, print_every=1000)
with torch.no_grad():
    u2 = model2(to_tensor(x_test2))[:, 0].numpy()
err2 = rel_l2(u2, np.cos(x_test2))
# trapezoidal reference: u = 1 + lam int_0^x K u with lam = -1 and K(x, t) = x - t
xs, us = volterra_trapezoid(lambda x, s: x - s, lambda x: np.ones_like(np.asarray(x, float)), -1.0, 0.0, B_END, 1001)
print(f"A-PINN error {err2:.2e}; trapezoidal rule (1001 nodes) error {rel_l2(us, np.cos(xs)):.2e}")

**Check your answer.** The A-PINN error should be below $10^{-2}$; the trapezoidal rule with 1001 nodes is far more accurate. Plot $A_\theta$ and $B_\theta$ against $1-\cos x$ and $\sin x$.

In [ ]:
assert err2 < 1e-2
print("Part 2 passed")

## Part 3 (optional): Filon weights

The module `chapters/ch07_integral_equations/_iecore.py` contains `filon_weights(theta, s=1.0)`. Verify numerically that the weights tend to the Simpson weights $(1/6, 2/3, 1/6)$ as $\theta\to0$, that $\alpha_0+\alpha_1+\alpha_2 = J_0(\theta)$, and compare `moments_closed` and `moments_series` for $\theta = 10^{-1}, 10^{-3}, 10^{-5}$. How many digits does the closed form lose?

In [ ]:
sys.path.insert(0, str(root / "chapters" / "ch07_integral_equations"))
from _iecore import filon_weights, moments_closed, moments_series
# TODO: your checks here

## Reflection

* Which of the three computations in this notebook would you hand to a PINN in practice, and why?
* What changes if the kernel contains an unknown parameter that must be identified from a few noisy values of $u$ (Chapter 11)?